# End-to-end verification: query the pipeline output with DuckDB

This notebook reads the Parquet files written by the pipeline directly from
MinIO/S3 and runs analytical queries over them. It's the smoke test that
proves the system works end-to-end — ingestion → quality → S3 → query.

**Prereqs:** `docker compose up --build` is running (MinIO on :9000, pipeline
ingesting from Bluesky firehose or demo mode).

```bash
pip install duckdb
```


In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
    INSTALL httpfs;
    LOAD httpfs;
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# Top tickers by mention volume across the full output
df = con.execute("""
    SELECT ticker,
           COUNT(*)                           AS mentions,
           ROUND(AVG(sentiment_compound), 3)  AS avg_sentiment,
           ROUND(STDDEV(sentiment_compound), 3) AS sentiment_stddev
    FROM read_parquet('s3://social-data/mentions/**/*.parquet')
    GROUP BY ticker
    ORDER BY mentions DESC
    LIMIT 15;
""").fetchdf()

print(df)


In [ ]:
# Volume + quality breakdown by source
# Useful for confirming live (bluesky) vs demo data are landing correctly,
# and that quality scoring is doing real work on real data.
df = con.execute("""
    SELECT source,
           COUNT(*)                           AS posts,
           ROUND(AVG(quality_score), 3)       AS avg_quality,
           ROUND(MIN(quality_score), 3)       AS min_quality,
           SUM(CASE WHEN tickers IS NOT NULL AND len(tickers) > 0 THEN 1 ELSE 0 END)
                                              AS posts_with_tickers
    FROM read_parquet('s3://social-data/posts/**/*.parquet')
    GROUP BY source;
""").fetchdf()

print(df)


In [ ]:
# Most recent 10 posts mentioning a specific ticker — the kind of query
# a model trainer would use to fetch the per-ticker time series.
df = con.execute("""
    SELECT to_timestamp(created_utc) AS ts,
           ticker,
           sentiment_compound,
           source,
           post_id
    FROM read_parquet('s3://social-data/mentions/**/*.parquet')
    WHERE ticker = 'NVDA'
    ORDER BY created_utc DESC
    LIMIT 10;
""").fetchdf()

print(df)


In [ ]:
# Hourly mention volume — shows the pipeline writing continuously over time.
# In live mode this should look like a smooth curve, with breaks if the
# pipeline was paused or the firehose dropped.
df = con.execute("""
    SELECT date_trunc('hour', to_timestamp(created_utc)) AS hour,
           COUNT(*) AS mentions,
           COUNT(DISTINCT ticker) AS distinct_tickers
    FROM read_parquet('s3://social-data/mentions/**/*.parquet')
    GROUP BY hour
    ORDER BY hour DESC
    LIMIT 24;
""").fetchdf()

print(df)


In [ ]:
# Sanity check: extreme sentiment posts. Useful for spot-checking that
# VADER scoring is reasonable on real Bluesky text (which is messier than
# the demo templates).
df = con.execute("""
    SELECT source,
           ROUND(sentiment_compound, 2) AS s,
           tickers,
           SUBSTR(body, 1, 120) AS preview
    FROM read_parquet('s3://social-data/posts/**/*.parquet')
    WHERE ABS(sentiment_compound) > 0.7
      AND len(tickers) > 0
    ORDER BY ABS(sentiment_compound) DESC
    LIMIT 10;
""").fetchdf()

print(df.to_string())
